In [ ]:
import seaborn as sns

df = sns.load_dataset("penguins").dropna()   # drop rows with missing data

print(df.shape)
print(df["species"].value_counts())
print(df.head())

In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import confusion_matrix, classification_report

device = "cuda" if torch.cuda.is_available() else "cpu"

FEATURES = ["bill_length_mm", "bill_depth_mm", "flipper_length_mm", "body_mass_g"]
species = df["species"].astype("category")
CLASSES = list(species.cat.categories)        # ['Adelie', 'Chinstrap', 'Gentoo']

X = df[FEATURES].values
y = species.cat.codes.values                  # species name → number 0, 1, 2

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# body mass (~4000 g) is much larger than bill length (~40 mm) → put them on the same scale
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

def to_tensor(X, y):
    return TensorDataset(torch.tensor(X, dtype=torch.float32), torch.tensor(y, dtype=torch.long))

train_loader = DataLoader(to_tensor(X_train, y_train), batch_size=16, shuffle=True)
test_loader = DataLoader(to_tensor(X_test, y_test), batch_size=16)

model = nn.Sequential(
    nn.Linear(4, 16), nn.ReLU(),   # 4 features in
    nn.Linear(16, 3),              # 3 species out
).to(device)

loss_fn = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-2)
EPOCHS = 30

# training
for epoch in range(EPOCHS):
    model.train()
    total_loss = 0
    for x, y in train_loader:
        x, y = x.to(device), y.to(device)
        loss = loss_fn(model(x), y)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    print(f"epoch {epoch+1} | loss {total_loss / len(train_loader):.4f}")

# evaluation
model.eval()
y_true, y_pred = [], []
with torch.no_grad():
    for x, y in test_loader:
        pred = model(x.to(device)).argmax(dim=1).cpu()
        y_true += y.tolist()
        y_pred += pred.tolist()

print(confusion_matrix(y_true, y_pred))
print(classification_report(y_true, y_pred, target_names=CLASSES))

In [ ]:
# predict a new penguin: bill length 45 mm, bill depth 15 mm, flipper 215 mm, mass 5000 g
penguin = scaler.transform([[45, 15, 215, 5000]])
with torch.no_grad():
    prob = torch.softmax(model(torch.tensor(penguin, dtype=torch.float32).to(device)), dim=1)[0]

for name, p in zip(CLASSES, prob):
    print(f"{name:10} {p:.0%}")